In [1]:
import pandas as pd
import numpy as np

In [2]:
df=pd.read_excel(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\2025_R_K_Puram, Delhi - DPCC.xlsx",skiprows=16)

In [3]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,Eth-Benzene,RH,WS,WD,SR,BP,AT,RF,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,205.88,281.68,26.30,44.99,44.00,1.45,12.72,0.07,0.49,NaN,83.97,0.76,193.65,40.24,NaN,12.31,NaN,0
1,02-01-2025 00:00,03-01-2025 00:00,205.92,289.21,29.26,42.00,43.74,1.69,12.93,0.00,0.12,NaN,85.34,0.69,193.24,40.52,NaN,12.42,NaN,0
2,03-01-2025 00:00,04-01-2025 00:00,289.75,384.37,72.57,53.40,86.05,2.92,14.02,0.01,0.18,NaN,83.85,0.34,188.89,40.93,767.20,14.07,NaN,0
3,04-01-2025 00:00,05-01-2025 00:00,278.92,359.71,41.68,57.14,63.09,2.71,13.66,0.01,0.19,NaN,86.27,0.66,174.42,43.00,770.21,14.18,NaN,0
4,05-01-2025 00:00,06-01-2025 00:00,174.17,245.00,7.24,39.55,26.61,1.24,12.61,0.00,0.10,NaN,84.45,0.80,169.90,48.34,749.45,13.61,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
315,12-11-2025 00:00,13-11-2025 00:00,388.74,546.72,56.84,80.04,86.67,2.26,46.38,0.23,4.60,NaN,67.39,0.60,197.58,52.68,985.53,19.34,NaN,0
316,13-11-2025 00:00,14-11-2025 00:00,318.77,454.09,40.62,93.20,82.60,2.21,38.72,0.23,4.28,NaN,68.33,0.49,194.91,50.45,985.96,18.89,NaN,0
317,14-11-2025 00:00,15-11-2025 00:00,232.08,363.54,31.61,85.16,70.99,1.84,48.65,0.37,3.35,NaN,65.35,0.50,204.89,60.17,986.00,19.09,NaN,0
318,15-11-2025 00:00,16-11-2025 00:00,266.54,401.75,39.56,81.11,75.31,3.11,52.44,0.15,3.71,NaN,64.44,0.50,197.73,65.71,986.00,19.14,NaN,0


In [4]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (320, 19)


In [5]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['Eth-Benzene']
Dropped rows (>70% NaN): 1
Missing values after imputation:
 From Date    0
To Date      0
PM2.5        0
PM10         0
NO           0
NO2          0
NOx          0
CO           0
Ozone        0
Benzene      0
Toluene      0
RH           0
WS           0
WD           0
SR           0
BP           0
AT           0
TOT-RF       0
dtype: int64


In [6]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [7]:

# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (319, 18)
          From Date           To Date   PM2.5    PM10     NO    NO2    NOx  \
0  01-01-2025 00:00  02-01-2025 00:00   68.00  281.68  26.30  44.99  44.00   
1  02-01-2025 00:00  03-01-2025 00:00   68.00  289.21  29.26  42.00  43.74   
2  03-01-2025 00:00  04-01-2025 00:00   68.00  384.37  11.93  53.40  86.05   
3  04-01-2025 00:00  05-01-2025 00:00   68.00  359.71  41.68  57.14  63.09   
4  05-01-2025 00:00  06-01-2025 00:00  174.17  245.00   7.24  39.55  26.61   

     CO  Ozone  Benzene  Toluene     RH    WS      WD     SR      BP     AT  \
0  1.45  12.72     0.07     0.49  83.97  0.76  193.65  40.24  985.78  26.41   
1  1.69  12.93     0.00     0.12  85.34  0.69  193.24  40.52  985.78  26.41   
2  0.97  14.02     0.01     0.18  83.85  0.34  188.89  40.93  985.78  14.07   
3  2.71  13.66     0.01     0.19  86.27  0.66  174.42  43.00  985.78  14.18   
4  1.24  12.61     0.00     0.10  84.45  0.80  169.90  48.34  985.78  13.61   

   TOT-RF  
0       0  
1       0

In [8]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [9]:
df

,From Date,To Date,PM2.5,PM10,NO,NO2,NOx,CO,Ozone,Benzene,Toluene,RH,WS,WD,SR,BP,AT,TOT-RF
0,01-01-2025 00:00,02-01-2025 00:00,-0.112946,1.233608,0.545320,0.445410,0.314969,0.681398,-1.355674,-0.739890,-0.205117,1.090491,-0.121043,0.390208,-1.190155,0.288150,0.208084,0.0
1,02-01-2025 00:00,03-01-2025 00:00,-0.112946,1.318712,0.726495,0.248981,0.303564,1.133813,-1.340998,-1.475498,-1.182903,1.186168,-0.379693,0.371530,-1.183564,0.288150,0.208084,0.0
2,03-01-2025 00:00,04-01-2025 00:00,-0.112946,2.394217,-0.334240,0.997907,2.159411,-0.223430,-1.264821,-1.370411,-1.024343,1.082111,-1.672944,0.173358,-1.173912,0.288150,-2.512958,0.0
3,04-01-2025 00:00,05-01-2025 00:00,-0.112946,2.115508,1.486699,1.243607,1.152315,3.056572,-1.289980,-1.370411,-0.997916,1.251117,-0.490543,-0.485847,-1.125186,0.288150,-2.488702,0.0
4,05-01-2025 00:00,06-01-2025 00:00,2.591403,0.819048,-0.621306,0.088027,-0.447810,0.285536,-1.363362,-1.475498,-1.235756,1.124013,0.026757,-0.691764,-0.999485,0.288150,-2.614391,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
314,12-11-2025 00:00,13-11-2025 00:00,-0.112946,-0.123318,2.414613,2.748028,2.186606,2.208296,0.996718,0.941499,-0.072984,-0.067414,-0.712243,0.569246,-0.897324,-0.320269,-1.350892,0.0
315,13-11-2025 00:00,14-11-2025 00:00,-0.112946,-0.123318,1.421819,-0.171469,2.008084,2.114043,0.461385,0.941499,-0.072984,-0.001767,-1.118693,0.447610,-0.949817,0.726212,-1.450120,0.0
316,14-11-2025 00:00,15-11-2025 00:00,-0.112946,2.158795,0.870334,-0.171469,1.498833,1.416571,1.155362,2.412714,-0.072984,-0.209882,-1.081743,0.902266,-0.721013,0.823559,-1.406019,0.0
317,15-11-2025 00:00,16-11-2025 00:00,-0.112946,2.590647,1.356938,2.818322,1.688322,-0.223430,1.420233,0.100804,-0.072984,-0.273434,-1.081743,0.576080,-0.590604,0.823559,-1.394993,0.0


In [10]:
df.to_excel('RKPuram2025.xlsx', index=False)